# Sex-Specific Models — Repeated Stratified 5-Fold CV (10 repeats x 5 folds, per sex)

Applies the same repeated-CV robustness check used for the pooled models (`05_repeated_cv_evaluation.ipynb`) to the sex-specific models, replacing a single random train/test split with a repeated cross-validation estimate for every male-only/female-only result.

`sex` itself is excluded from the feature set (constant within each sex-specific model).

In [1]:
import ast
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier, BaggingClassifier, ExtraTreesClassifier,
                               AdaBoostClassifier, StackingClassifier)
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
HORIZONS = [1,2,3,4,5]
SEEDS = list(range(10))
SEXES = ['M', 'F']
MIN_PATIENTS = 30  # skip a model/sex/horizon combo if too few patients to fold reliably

pd.set_option('display.max_columns', None)

## 1. Load modeling dataset + feature list (sex excluded -- constant within each sex-specific model)

In [2]:
df = pd.read_csv('modeling_dataset.csv')
COMORBID_FEATURE_COLS = [c for c in df.columns if c not in (
    ['person_id', 'sex', 'age', 'postcode', 'rurality', 'ses_irsd_decile', 'ses_missing',
     'incident_cvd', 'years_followup', 'split',
     'baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
     'baseline_n_episodes', 'baseline_history_days']
    + [f'label_{h}y' for h in HORIZONS] + [f'eligible_{h}y' for h in HORIZONS]
)]
IS_GEO = 'rurality' in df.columns
if IS_GEO:
    df['rurality'] = df['rurality'].fillna('Unknown')

UTILISATION_COLS = ['baseline_n_diagnoses', 'baseline_n_procedures', 'baseline_n_claims',
                     'baseline_n_episodes', 'baseline_history_days']
if IS_GEO:
    NUMERIC_COLS = ['age'] + UTILISATION_COLS + ['ses_irsd_decile', 'ses_missing']
    CATEGORICAL_COLS = ['rurality']
else:
    NUMERIC_COLS = ['age'] + UTILISATION_COLS
    CATEGORICAL_COLS = []

FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS + COMORBID_FEATURE_COLS
print(f'{len(FEATURE_COLS)} features ({len(COMORBID_FEATURE_COLS)} comorbidity-derived)')

def make_preprocessor(scaled):
    num_step = StandardScaler() if scaled else 'passthrough'
    transformers = []
    if CATEGORICAL_COLS:
        transformers.append(('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), CATEGORICAL_COLS))
    transformers.append(('num', num_step, NUMERIC_COLS))
    transformers.append(('bin', 'passthrough', COMORBID_FEATURE_COLS))
    return ColumnTransformer(transformers)

sex_specific_results = pd.read_csv('sex_specific_results.csv')

def load_best_params(model_name, sex_val, h):
    row = sex_specific_results[
        (sex_specific_results['model'] == model_name) &
        (sex_specific_results['sex'] == sex_val) &
        (sex_specific_results['horizon'] == f'{h}y')
    ]
    if len(row) == 0:
        return None
    params = ast.literal_eval(row.iloc[0]['best_params'])
    return {k.replace('clf__', ''): v for k, v in params.items()}

18 features (12 comorbidity-derived)


## 2. Model configs

In [3]:
MODEL_CONFIGS = {
    'Logistic Regression': dict(cls=LogisticRegression,
                                 fixed_kwargs=dict(solver='saga', max_iter=5000), scaled=True, has_seed=True),
    'Random Forest':       dict(cls=RandomForestClassifier,
                                 fixed_kwargs=dict(n_jobs=-1), scaled=False, has_seed=True),
    'XGBoost':             dict(cls=XGBClassifier,
                                 fixed_kwargs=dict(eval_metric='logloss'), scaled=False, has_seed=True,
                                 needs_scale_pos_weight=True),
    'LightGBM':            dict(cls=LGBMClassifier,
                                 fixed_kwargs=dict(class_weight='balanced', verbose=-1), scaled=False, has_seed=True),
    'SVM':                 dict(cls=SVC,
                                 fixed_kwargs=dict(probability=True), scaled=True, has_seed=True),
    'Decision Tree':       dict(cls=DecisionTreeClassifier,
                                 fixed_kwargs=dict(), scaled=False, has_seed=True),
    'Naive Bayes':         dict(cls=GaussianNB,
                                 fixed_kwargs=dict(), scaled=True, has_seed=False),
    'KNN':                 dict(cls=KNeighborsClassifier,
                                 fixed_kwargs=dict(), scaled=True, has_seed=False),
    'Bagging (DT base)':   dict(cls=BaggingClassifier,
                                 base_kwargs=dict(estimator=DecisionTreeClassifier(random_state=RANDOM_STATE)),
                                 fixed_kwargs=dict(n_jobs=-1), scaled=False, has_seed=True),
    'Extra Trees':         dict(cls=ExtraTreesClassifier,
                                 fixed_kwargs=dict(n_jobs=-1), scaled=False, has_seed=True),
    'Bagging (KNN base)':  dict(cls=BaggingClassifier,
                                 base_kwargs=dict(estimator=KNeighborsClassifier()),
                                 fixed_kwargs=dict(n_jobs=-1), scaled=True, has_seed=True),
    'AdaBoost':            dict(cls=AdaBoostClassifier,
                                 fixed_kwargs=dict(), scaled=False, has_seed=True),
    'Stacking (diverse)':  dict(cls=StackingClassifier,
                                 base_kwargs=dict(
                                     estimators=[
                                         ('lr', LogisticRegression(solver='saga', max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE)),
                                         ('rf', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
                                         ('svm', SVC(probability=True, class_weight='balanced', random_state=RANDOM_STATE)),
                                     ],
                                     final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), cv=5),
                                 fixed_kwargs=dict(n_jobs=-1), scaled=True, has_seed=False),
    'Stacking (boosting)': dict(cls=StackingClassifier,
                                 base_kwargs=dict(
                                     estimators=[
                                         ('xgb', XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')),
                                         ('lgbm', LGBMClassifier(class_weight='balanced', random_state=RANDOM_STATE, verbose=-1)),
                                         ('ada', AdaBoostClassifier(random_state=RANDOM_STATE)),
                                     ],
                                     final_estimator=LogisticRegression(max_iter=5000, random_state=RANDOM_STATE), cv=5),
                                 fixed_kwargs=dict(n_jobs=-1), scaled=True, has_seed=False),
}

## 3. Run all 14 models x 2 sexes x horizons x 10 repeats x 5 folds

In [4]:
all_summaries = []
all_raw = []

for model_name, cfg in MODEL_CONFIGS.items():
    preprocessor = make_preprocessor(cfg['scaled'])

    for sex_val in SEXES:
        for h in HORIZONS:
            elig_col, label_col = f'eligible_{h}y', f'label_{h}y'
            sub = df[(df[elig_col]) & (df['sex'] == sex_val)].copy()
            X_all, y_all = sub[FEATURE_COLS], sub[label_col].astype(int)

            if len(X_all) < MIN_PATIENTS or y_all.nunique() < 2:
                print(f'{model_name} / {sex_val} / {h}y: skipped (n={len(X_all)})')
                continue

            base_params = load_best_params(model_name, sex_val, h)
            if base_params is None:
                print(f'{model_name} / {sex_val} / {h}y: skipped (no saved best_params for this combination)')
                continue
            base_params.update(cfg['fixed_kwargs'])

            fold_metrics = []
            for seed in SEEDS:
                skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
                for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
                    X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
                    y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]

                    if y_test.nunique() < 2:
                        continue

                    run_params = dict(base_params)
                    if cfg.get('needs_scale_pos_weight'):
                        run_params['scale_pos_weight'] = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
                    if cfg['has_seed']:
                        run_params['random_state'] = seed

                    clf_obj = cfg['cls'](**cfg.get('base_kwargs', {}))
                    pipe = Pipeline([('preprocess', preprocessor), ('clf', clf_obj)])
                    pipe.set_params(**{f'clf__{k}': v for k, v in run_params.items()})
                    pipe.fit(X_train, y_train)
                    proba = pipe.predict_proba(X_test)[:, 1]
                    pred = (proba >= 0.5).astype(int)

                    row = dict(model=model_name, sex=sex_val, horizon=f'{h}y', seed=seed, fold=fold_idx,
                               auc=roc_auc_score(y_test, proba),
                               accuracy=accuracy_score(y_test, pred),
                               precision=precision_score(y_test, pred, zero_division=0),
                               recall=recall_score(y_test, pred, zero_division=0),
                               f1=f1_score(y_test, pred, zero_division=0))
                    fold_metrics.append(row)
                    all_raw.append(row)

            if not fold_metrics:
                print(f'{model_name} / {sex_val} / {h}y: no valid folds (all skipped)')
                continue

            mdf = pd.DataFrame(fold_metrics)
            summary = dict(model=model_name, sex=sex_val, horizon=f'{h}y', n_evaluations=len(fold_metrics))
            for metric in ['auc', 'accuracy', 'precision', 'recall', 'f1']:
                summary[f'{metric}_mean'] = round(mdf[metric].mean(), 3)
                summary[f'{metric}_std'] = round(mdf[metric].std(), 3)
            all_summaries.append(summary)
            print(f"{model_name:20s} {sex_val} {h}y: AUC {summary['auc_mean']:.3f} +/- {summary['auc_std']:.3f} (n={len(fold_metrics)})")

Logistic Regression  M 1y: AUC 0.672 +/- 0.053 (n=50)


Logistic Regression  M 2y: AUC 0.656 +/- 0.042 (n=50)


Logistic Regression  M 3y: AUC 0.657 +/- 0.035 (n=50)


Logistic Regression  M 4y: AUC 0.657 +/- 0.037 (n=50)


Logistic Regression  M 5y: AUC 0.623 +/- 0.051 (n=50)


...

Stacking (boosting)  F 1y: AUC 0.780 +/- 0.057 (n=50)


Stacking (boosting)  F 2y: AUC 0.773 +/- 0.047 (n=50)


Stacking (boosting)  F 3y: AUC 0.750 +/- 0.052 (n=50)


Stacking (boosting)  F 4y: AUC 0.749 +/- 0.042 (n=50)


Stacking (boosting)  F 5y: AUC 0.701 +/- 0.059 (n=50)


## 4. Save results

In [5]:
raw_df = pd.DataFrame(all_raw)
raw_df.to_csv('sex_specific_repeated_cv_raw_per_fold.csv', index=False)

summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv('sex_specific_repeated_cv_summary.csv', index=False)
summary_df

,model,sex,horizon,n_evaluations,auc_mean,auc_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,Logistic Regression,M,1y,50,0.672,0.053,0.899,0.005,0.574,0.388,0.046,0.036,0.083,0.062
1,Logistic Regression,M,2y,50,0.656,0.042,0.853,0.005,0.456,0.408,0.022,0.020,0.042,0.038
2,Logistic Regression,M,3y,50,0.657,0.035,0.628,0.032,0.306,0.033,0.562,0.075,0.395,0.042
3,Logistic Regression,M,4y,50,0.657,0.037,0.715,0.017,0.566,0.137,0.133,0.045,0.213,0.063
4,Logistic Regression,M,5y,50,0.623,0.051,0.585,0.040,0.551,0.052,0.495,0.052,0.520,0.041
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,Stacking (boosting),F,1y,50,0.780,0.057,0.906,0.008,0.406,0.259,0.086,0.056,0.137,0.084
136,Stacking (boosting),F,2y,50,0.773,0.047,0.861,0.012,0.519,0.223,0.127,0.061,0.195,0.083
137,Stacking (boosting),F,3y,50,0.750,0.052,0.812,0.015,0.583,0.231,0.112,0.062,0.180,0.087
138,Stacking (boosting),F,4y,50,0.749,0.042,0.755,0.023,0.659,0.157,0.188,0.064,0.286,0.081


## 5. Male vs. female AUC comparison (repeated-CV)

In [6]:
piv = summary_df.pivot_table(index=['model', 'horizon'], columns='sex', values='auc_mean')
piv['gap_M_minus_F'] = (piv['M'] - piv['F']).round(3)
piv.to_csv('sex_specific_repeated_cv_gap.csv')
piv

sex                   F      M  gap_M_minus_F
model    horizon                             
AdaBoost 1y       0.775  0.670         -0.105
         2y       0.773  0.667         -0.106
         3y       0.759  0.669         -0.090
         4y       0.759  0.656         -0.103
         5y       0.715  0.632         -0.083
...                 ...    ...            ...
XGBoost  1y       0.765  0.677         -0.088
         2y       0.769  0.663         -0.106
         3y       0.747  0.665         -0.082
         4y       0.730  0.662         -0.068
         5y       0.702  0.650         -0.052

[70 rows x 3 columns]